# Bài 7: Comprehension, Generator, Lambda

Ba công cụ giúp viết code Python **ngắn gọn và đúng "chất Python"**. Bạn sẽ gặp chúng liên tục khi xử lý dữ liệu: lọc mẫu, biến đổi nhãn, sắp xếp theo điều kiện, và đặc biệt là `generator` — nền tảng của việc đọc dữ liệu theo từng batch (DataLoader) mà không tốn RAM.

Mục tiêu bài học:
- List comprehension (có điều kiện `if`, có `if/else`)
- Dict comprehension và set comprehension
- Generator expression và hàm generator (`yield`) — tính toán "lười" (lazy)
- `lambda` và dùng với `sorted(key=...)`, `map`, `filter`

> Nhắc lại cách học: đọc lý thuyết → chạy thử ví dụ → tự làm bài tập, đừng xem lời giải trước.

## 1. List comprehension

Cú pháp: `[biểu_thức for phần_tử in iterable]` — tạo list mới trong **một dòng**, thay cho vòng `for` + `append` (bài 4).

In [1]:
numbers = [1, 2, 3, 4, 5]

# Cách cũ: for + append
squares_old = []
for n in numbers:
    squares_old.append(n ** 2)

# Cách mới: list comprehension
squares = [n ** 2 for n in numbers]

print(squares_old)
print(squares)

# Ví dụ thực tế: chuẩn hoá nhãn chữ
labels = ["Cat", "DOG", "Bird"]
print([label.lower() for label in labels])

[1, 4, 9, 16, 25]
[1, 4, 9, 16, 25]
['cat', 'dog', 'bird']


## 2. Thêm điều kiện

- **Lọc**: `[x for x in data if điều_kiện]` — `if` đứng **sau** `for`, chỉ giữ phần tử thoả điều kiện.
- **Biến đổi theo điều kiện**: `[a if điều_kiện else b for x in data]` — `if/else` đứng **trước** `for`, mọi phần tử đều được giữ.

In [2]:
scores = [0.45, 0.91, 0.72, 0.30, 0.88]

# Lọc: chỉ giữ score >= 0.7
good = [s for s in scores if s >= 0.7]
print(good)

# if/else: gán nhãn pass/fail cho từng score
status = ["pass" if s >= 0.7 else "fail" for s in scores]
print(status)

# Kết hợp với enumerate: lấy chỉ số của các score tốt
good_idx = [i for i, s in enumerate(scores) if s >= 0.7]
print(good_idx)

[0.91, 0.72, 0.88]
['fail', 'pass', 'pass', 'fail', 'pass']
[1, 2, 4]


## 3. Dict & set comprehension

Cùng ý tưởng, dùng `{}`:
- Dict: `{key: value for ... in ...}`
- Set: `{biểu_thức for ... in ...}` (tự loại trùng)

In [3]:
class_names = ["cat", "dog", "bird"]

# class_to_idx của bài 5, giờ chỉ một dòng
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
print(class_to_idx)

# đảo ngược: idx -> name (rất hay dùng để giải mã dự đoán của model)
idx_to_class = {idx: name for name, idx in class_to_idx.items()}
print(idx_to_class)

# set comprehension: độ dài các từ, không trùng
words = ["ai", "ml", "python", "data", "code"]
print({len(w) for w in words})

{'cat': 0, 'dog': 1, 'bird': 2}
{0: 'cat', 1: 'dog', 2: 'bird'}
{2, 4, 6}


### Comprehension lồng nhau (làm phẳng ma trận)

Thứ tự `for` đọc **từ trái sang phải giống như vòng for lồng nhau viết ra**. Đừng lạm dụng — nếu khó đọc, hãy dùng vòng `for` thường.

In [4]:
matrix = [[1, 2, 3], [4, 5, 6]]

flat = [value for row in matrix for value in row]
print(flat)

# Tạo ma trận 3x3 toàn số 0 (bài 4: nested list)
zeros = [[0 for _ in range(3)] for _ in range(3)]
print(zeros)

[1, 2, 3, 4, 5, 6]
[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


## 4. Generator — tính toán "lười" (lazy)

List comprehension tạo **toàn bộ list trong bộ nhớ**. Nếu dữ liệu có hàng triệu phần tử, rất tốn RAM. **Generator expression** dùng `()` thay `[]`: không tính trước, chỉ sinh từng giá trị **khi được yêu cầu**.

In [5]:
import sys

big_list = [x * 2 for x in range(1_000_000)]   # lưu cả triệu số trong RAM
big_gen = (x * 2 for x in range(1_000_000))     # chưa tính gì cả

print("list:", sys.getsizeof(big_list), "bytes")
print("generator:", sys.getsizeof(big_gen), "bytes")

# Lấy giá trị bằng next() hoặc vòng for
small_gen = (x * 2 for x in range(3))
print(next(small_gen))
print(next(small_gen))
print(list(small_gen))   # phần còn lại

# Dùng thẳng với sum/max/min -- không cần tạo list trung gian
print(sum(x * x for x in range(10)))

list: 8448728 bytes
generator: 200 bytes
0
2
[4]
285


⚠️ **Generator chỉ duyệt được một lần.** Dùng hết rồi là cạn, muốn duyệt lại phải tạo generator mới.

In [6]:
gen = (x for x in range(3))
print(list(gen))   # [0, 1, 2]
print(list(gen))   # [] -- đã cạn

[0, 1, 2]
[]


## 5. Hàm generator với `yield`

Hàm có `yield` thay cho `return` sẽ trở thành **generator**: mỗi lần `yield` nó trả một giá trị rồi **tạm dừng**, lần sau chạy tiếp từ chỗ đó. Ví dụ kinh điển trong AI: **chia dữ liệu thành các batch**.

In [7]:
def batches(data, batch_size):
    """Chia data thành từng batch kích thước batch_size."""
    for start in range(0, len(data), batch_size):
        yield data[start:start + batch_size]

samples = list(range(10))
for batch in batches(samples, batch_size=4):
    print(batch)

# Đây chính là ý tưởng cốt lõi của DataLoader trong PyTorch (Module 6)

[0, 1, 2, 3]
[4, 5, 6, 7]
[8, 9]


## 6. Lambda — hàm ẩn danh một dòng

`lambda tham_số: biểu_thức` tạo hàm nhỏ không cần đặt tên, tự động `return` biểu thức. Chỉ nên dùng cho hàm **ngắn, dùng một lần**; hàm phức tạp thì dùng `def` (bài 6).

In [8]:
square = lambda x: x ** 2        # tương đương def square(x): return x ** 2
add = lambda a, b: a + b

print(square(4))
print(add(2, 3))

16
5


### Ứng dụng quan trọng nhất: `sorted(..., key=...)`, `max(..., key=...)`

`key` nhận một hàm; Python gọi hàm đó trên từng phần tử và so sánh theo kết quả.

In [9]:
results = [
    {"model": "resnet18", "acc": 0.89},
    {"model": "resnet50", "acc": 0.93},
    {"model": "vgg16", "acc": 0.91},
]

# Sắp xếp các model theo accuracy giảm dần
ranked = sorted(results, key=lambda r: r["acc"], reverse=True)
for r in ranked:
    print(r["model"], r["acc"])

# Model tốt nhất
best = max(results, key=lambda r: r["acc"])
print("Tốt nhất:", best["model"])

# Sắp xếp chuỗi theo độ dài
print(sorted(["python", "ai", "data"], key=lambda w: len(w)))

resnet50 0.93
vgg16 0.91
resnet18 0.89
Tốt nhất: resnet50
['ai', 'data', 'python']


## 7. `map` và `filter`

- `map(hàm, iterable)`: áp hàm lên từng phần tử.
- `filter(hàm, iterable)`: giữ phần tử mà hàm trả `True`.

Cả hai trả về **iterator lười** (giống generator) nên cần bọc `list(...)`. Trong thực tế, list comprehension thường **dễ đọc hơn** — nhưng bạn sẽ gặp `map`/`filter` trong code người khác.

In [10]:
nums = [1, 2, 3, 4, 5, 6]

print(list(map(lambda x: x * 10, nums)))
print(list(filter(lambda x: x % 2 == 0, nums)))

# Tương đương bằng comprehension (thường được ưa chuộng hơn)
print([x * 10 for x in nums])
print([x for x in nums if x % 2 == 0])

[10, 20, 30, 40, 50, 60]
[2, 4, 6]
[10, 20, 30, 40, 50, 60]
[2, 4, 6]


## Bài tập thực hành

1. Dùng list comprehension tạo list các số chẵn từ 1 đến 20 (`range(1, 21)`), rồi in ra.
2. Cho `texts = ["  Hello ", "AI  ", " python"]`. Dùng comprehension tạo list đã `.strip()` và `.lower()`: `["hello", "ai", "python"]`.
3. Cho `scores = [0.45, 0.91, 0.72, 0.30, 0.88]`. Dùng comprehension tạo list `predictions` gồm `1` nếu `score >= 0.5`, ngược lại `0`. (Đây chính là bước chuyển xác suất thành nhãn!)
4. Cho `words = ["apple", "banana", "cherry"]`. Dùng dict comprehension tạo dict `{từ: độ dài}`, rồi in ra.
5. Viết **hàm generator** `countdown(n)` yield lần lượt `n, n-1, ..., 1`. Dùng `for` in kết quả của `countdown(5)`.
6. Cho `students = [("An", 8.5), ("Bình", 9.0), ("Chi", 7.0)]`. Dùng `sorted` + `lambda` để sắp xếp theo điểm **giảm dần** và in ra tên người đứng đầu.
7. **(Nâng cao)** Cho `matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]`. Dùng comprehension lồng nhau tạo **ma trận chuyển vị** (transpose): `[[1, 4, 7], [2, 5, 8], [3, 6, 9]]`. Gợi ý: phần tử `[i][j]` của kết quả là `matrix[j][i]`.

In [11]:
# Bài 1: Số chẵn từ 1 đến 20

In [12]:
# Bài 2: strip + lower

In [13]:
# Bài 3: score -> predictions

In [14]:
# Bài 4: dict từ -> độ dài

In [15]:
# Bài 5: generator countdown

In [16]:
# Bài 6: sorted + lambda

In [17]:
# Bài 7 (nâng cao): transpose ma trận

## Gợi ý

- Bài 1: `[n for n in range(1, 21) if n % 2 == 0]`.
- Bài 3: dạng `[1 if s >= 0.5 else 0 for s in scores]` (nhớ: `if/else` đứng **trước** `for`).
- Bài 5: dùng `for i in range(n, 0, -1): yield i`.
- Bài 6: `sorted(students, key=lambda s: s[1], reverse=True)[0][0]`.
- Bài 7: `[[matrix[j][i] for j in range(3)] for i in range(3)]`.

## Lời giải mẫu

⚠️ Chỉ chạy ô dưới đây **sau khi** bạn đã tự làm xong 7 bài tập ở trên.

In [18]:
# --- Lời giải mẫu ---

# Bài 1
evens = [n for n in range(1, 21) if n % 2 == 0]
print(evens)

# Bài 2
texts = ["  Hello ", "AI  ", " python"]
cleaned = [t.strip().lower() for t in texts]
print(cleaned)

# Bài 3
scores = [0.45, 0.91, 0.72, 0.30, 0.88]
predictions = [1 if s >= 0.5 else 0 for s in scores]
print(predictions)

# Bài 4
words = ["apple", "banana", "cherry"]
lengths = {w: len(w) for w in words}
print(lengths)

# Bài 5
def countdown(n):
    for i in range(n, 0, -1):
        yield i

for value in countdown(5):
    print(value)

# Bài 6
students = [("An", 8.5), ("Bình", 9.0), ("Chi", 7.0)]
ranked = sorted(students, key=lambda s: s[1], reverse=True)
print(ranked)
print("Đứng đầu:", ranked[0][0])

# Bài 7
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
transposed = [[matrix[j][i] for j in range(3)] for i in range(3)]
print(transposed)

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
['hello', 'ai', 'python']
[0, 1, 1, 0, 1]
{'apple': 5, 'banana': 6, 'cherry': 6}
5
4
3
2
1
[('Bình', 9.0), ('An', 8.5), ('Chi', 7.0)]
Đứng đầu: Bình
[[1, 4, 7], [2, 5, 8], [3, 6, 9]]


## Tổng kết

Bạn đã học được:
- ✅ List/dict/set comprehension, có điều kiện lọc (`if`) và `if/else`
- ✅ Generator expression `( ... )` và hàm generator với `yield` — tính toán lười, tiết kiệm RAM, nền tảng của việc chia batch
- ✅ `lambda` và dùng với `sorted(key=...)`, `max(key=...)`
- ✅ `map`, `filter` (và vì sao comprehension thường dễ đọc hơn)

**Bài tiếp theo:** Bài 8 — Lập trình hướng đối tượng (OOP): `class`, `__init__`, thuộc tính, phương thức, kế thừa — nền tảng để hiểu `nn.Module` của PyTorch và `Estimator` của scikit-learn. Làm xong bài tập ở trên rồi báo mình nhé!